# feral — scipy.sparse & numpy interop

`feral.from_scipy` / `feral.to_scipy` round-trip a symmetric `scipy.sparse` matrix into feral's lower-triangular CSC and back. This notebook shows the conversion and validates feral's solve against `scipy.sparse.linalg.spsolve` and `numpy.linalg.solve`.

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import feral

rng = np.random.default_rng(7)

## A random symmetric indefinite matrix

Half the diagonal shifted positive, half negative, to make it genuinely indefinite (a stress case for a direct solver).

In [ ]:
def random_sym_indef(n, density=0.2, seed=0):
    r = np.random.default_rng(seed)
    Ad = sp.random(n, n, density=density, format='csc',
                   random_state=r).toarray()
    Ad = (Ad + Ad.T) / 2.0
    shift = np.empty(n)
    shift[: n // 2] = 5.0
    shift[n // 2 :] = -5.0
    Ad += np.diag(shift)
    return sp.csc_matrix(Ad)

n = 60
A_sp = random_sym_indef(n, density=0.15, seed=11)
print('shape =', A_sp.shape, ' nnz(full) =', A_sp.nnz)

## Convert into feral

`symmetric='full'` tells feral the scipy matrix stores the full symmetric matrix; it reads the lower triangle.

In [ ]:
A = feral.from_scipy(A_sp, symmetric='full')
print('feral:', A, ' nnz(lower) =', A.nnz)

## Factor, report inertia, solve

In [ ]:
solver = feral.Solver()
status, inertia = solver.factor(A)
print('status :', feral.FactorStatus(status).name)
print('inertia:', inertia)

b = rng.standard_normal(n)
x_feral = solver.solve_refined(A, b)

## Compare against scipy and numpy reference solves

In [ ]:
x_spsolve = spla.spsolve(A_sp.tocsc(), b)
x_dense = np.linalg.solve(A_sp.toarray(), b)

print('‖feral - spsolve‖inf :', f'{np.max(np.abs(x_feral - x_spsolve)):.3e}')
print('‖feral - dense‖inf   :', f'{np.max(np.abs(x_feral - x_dense)):.3e}')
assert np.allclose(x_feral, x_dense, atol=1e-9, rtol=1e-9)

## Round-trip feral -> scipy

In [ ]:
A_back = feral.to_scipy(A)
# A_back mirrors the lower triangle to a full symmetric matrix
diff = np.max(np.abs(A_back.toarray() - A_sp.toarray()))
print(f'max |round-trip - original| = {diff:.3e}')
assert diff < 1e-12